<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Impedimentos Sociais, Ambientais e Climáticos</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Thales Sehn Körting, Gilberto Queiroz, Karine Ferreira, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 11 de abril de 2026
</div>


<br/>

</div>

Esse notebook aborda de forma simplificada as restrições legais para o acesso ao crédito rural com base em critérios sociais, ambientais e climáticos.

# <span style="color:#336699">Contextualização</span>
<hr style="border:1px solid #0077b9;">

Com as recentes atualizações do Conselho Monetário Nacional (como as Resoluções CMN Nº 5.193/2024, 5.267/2025 e 5.268/2025), a verificação de financiamentos tornou-se mais rigorosa. Hoje, exige-se a identificação da propriedade via Cadastro Ambiental Rural (CAR) e o monitoramento obrigatório por sensoriamento remoto para áreas superiores a 300 hectares, garantindo que os recursos financiados respeitem a conformidade socioambiental.

Na atividade prática de hoje, vamos analisar os impedimentos definidos para esses empreendimentos.

# <span style="color:#336699">3 - Unidades de Conservação</span>
<hr style="border:1px solid #0077b9;">

As áreas protegidas são espaços territorialmente demarcados, tendo como principal função a conservação e/ou a preservação de recursos naturais e/ou culturais, a elas associados. Elas podem ser definidas como uma área terrestre e/ou marinha especialmente dedicada à proteção e manutenção da diversidade biológica e dos recursos naturais e culturais associados, manejados através de instrumentos legais ou outros instrumentos efetivos. Desta maneira, as áreas protegidas foram criadas com o objetivo de preservar e conservar o patrimônio natural de nossos ecossistemas, garantindo seus aspectos ecológicos, geológicos, históricos e culturais.

De acordo com a Resolução CMN Nº 5.193/2024:

> 5 - Não será concedido crédito rural para empreendimento situado em imóvel rural total ou parcialmente inserido em
Unidade de Conservação, desde que registrada no Cadastro Nacional de Unidades de Conservação (CNUC) do
Ministério do Meio Ambiente e Mudança do Clima (MMA), salvo se a atividade econômica se encontrar em
conformidade com o Plano de Manejo da Unidade de Conservação, respeitadas as disposições do art. 28 da Lei
nº 9.985, de 18 de julho de 2000, e as disposições específicas aplicáveis à população tradicional beneficiária ou
residente, na forma do Decreto nº 4.340, de 22 de agosto de 2002. (Res CMN 5.193 art 1º)

> 5-A - Até 30 de junho de 2028, com base nos arts. 17, § 2º, 18, caput, e 20, § 1º, da Lei nº 9.985, de 18 julho de
2000, na ausência do Plano de Manejo publicado para Reserva Extrativista – RESEX, Floresta Nacional e
Reserva de Desenvolvimento Sustentável, será admitida, para a concessão do crédito rural, a anuência publicada
no sítio eletrônico oficial do órgão ambiental responsável pela gestão da Unidade de Conservação – UC, emitida
para povos e comunidades tradicionais beneficiários da respectiva UC, desde que: (Res CMN 5.268 art 1º)
a) as operações sejam contratadas no âmbito do Programa Nacional de Fortalecimento da Agricultura Familiar –
Pronaf; e
b) as atividades produtivas destinadas à implementação de práticas sustentáveis sejam compatíveis com os
objetivos de criação da Unidade.

> 6 - No caso de Unidade de Conservação de domínio exclusivamente público, o impedimento de que trata o item 5 se
aplica apenas a empreendimento inserido total ou parcialmente em imóvel cujo processo de regularização
fundiária tenha sido concluído, nos termos da regulamentação aplicável. (Res CMN 5.193 art 1º)


Uma representação ilustrativa em forma de diagrama da restrição pode ser observada na Figura 3:

![car](https://i.imgur.com/Ompbm6B.png "Unidades de Conservação")

Prática: vamos simular a verificação de sobreposição.




**Configuração e Exemplo atende a norma**

In [ ]:
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import requests, zipfile, io
import geopandas as gpd

In [ ]:
imovel_rural = {
    "geometria": Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]),
    "beneficiario": "João da Silva",
    "atividade": "soja"
}

In [ ]:
lista_uc = [
    {"geometria": Polygon([(5, 5), (7, 5), (7, 7), (5, 7)]), "plano_manejo": False, "atividade": "milho", "gestor_permite": False},
    {"geometria": Polygon([(10, 10), (12, 10), (12, 12), (10, 12)]), "plano_manejo": True, "atividade": "soja", "gestor_permite": False},
    {"geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]), "plano_manejo": False, "atividade": "vazio", "gestor_permite": True}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_imovel, y_imovel = imovel_rural["geometria"].exterior.xy

ax.plot(x_imovel, y_imovel, color='#1f77b4', linewidth=2, label='Imóvel Rural')
ax.fill(x_imovel, y_imovel, color='#1f77b4', alpha=0.3)

for i, uc in enumerate(lista_uc):
    x_uc, y_uc = uc["geometria"].exterior.xy

    label = 'Unidade de Conservação (UC)' if i == 0 else None

    ax.plot(x_uc, y_uc, color='#d62728', linewidth=2, label=label)
    ax.fill(x_uc, y_uc, color='#d62728', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Imóvel Rural vs. Unidades de Conservação (UC)')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, uc in enumerate(lista_uc):

    # Condição 1: Há sobreposição?
    if imovel_rural["geometria"].intersects(uc["geometria"]):

        # Condição 2: A UC tem plano de manejo?
        if uc["plano_manejo"]:

            # Condição 3: A atividade é contemplada no plano de manejo da UC?
            if uc["atividade"] == imovel_rural["atividade"]:
                pass # Atende a norma nesta UC específica
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"A atividade ('{imovel_rural['atividade']}') não é contemplada no plano de manejo da UC {i}."
                break

        # Caso a unidade NÃO tenha plano de manejo
        else:

            # Condição 4: O gestor da UC permite a atividade?
            if uc["gestor_permite"]:
                pass # Atende a norma nesta UC específica
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"A UC {i} não possui plano de manejo e o gestor NÃO permitiu a atividade."
                break

    # Se não há sobreposição, atende a norma (passa direto)
    else:
        pass

# Validação Final
if imovel_atende_norma:
    print("Resultado: O imóvel rural atende a norma.")
else:
    print(f"Resultado: O imóvel rural não atende a norma porque houve {motivo_impedimento}")

**Configuração e Exemplo atende a norma**

In [ ]:
imovel_rural = {
    "geometria": Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]),
    "beneficiario": "João da Silva",
    "atividade": "soja"
}

In [ ]:
lista_uc = [
    {"geometria": Polygon([(10, 10), (12, 10), (12, 12), (10, 12)]), "plano_manejo": True, "atividade": "soja", "gestor_permite": False},
    {"geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]), "plano_manejo": False, "atividade": "vazio", "gestor_permite": True},
    {"geometria": Polygon([(0, 0), (1, 0), (1, 1), (0, 1)]), "plano_manejo": True,  "atividade": "milho", "gestor_permite": True}
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

x_imovel, y_imovel = imovel_rural["geometria"].exterior.xy

ax.plot(x_imovel, y_imovel, color='#1f77b4', linewidth=2, label='Imóvel Rural')
ax.fill(x_imovel, y_imovel, color='#1f77b4', alpha=0.3)

for i, uc in enumerate(lista_uc):
    x_uc, y_uc = uc["geometria"].exterior.xy

    label = 'Unidade de Conservação (UC)' if i == 0 else None

    ax.plot(x_uc, y_uc, color='#d62728', linewidth=2, label=label)
    ax.fill(x_uc, y_uc, color='#d62728', alpha=0.3)

ax.set_aspect('equal')
ax.set_title('Visualização: Imóvel Rural vs. Unidades de Conservação (UC)')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
ax.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
imovel_atende_norma = True
motivo_impedimento = ""

for i, uc in enumerate(lista_uc):

    # Condição 1: Há sobreposição?
    if imovel_rural["geometria"].intersects(uc["geometria"]):

        # Condição 2: A UC tem plano de manejo?
        if uc["plano_manejo"]:

            # Condição 3: A atividade é contemplada no plano de manejo da UC?
            if uc["atividade"] == imovel_rural["atividade"]:
                pass # Atende a norma nesta UC específica
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"A atividade ('{imovel_rural['atividade']}') não é contemplada no plano de manejo da UC {i}."
                break

        # Caso a unidade NÃO tenha plano de manejo
        else:

            # Condição 4: O gestor da UC permite a atividade?
            if uc["gestor_permite"]:
                pass # Atende a norma nesta UC específica
            else:
                imovel_atende_norma = False
                motivo_impedimento = f"A UC {i} não possui plano de manejo e o gestor NÃO permitiu a atividade."
                break

    # Se não há sobreposição, atende a norma (passa direto)
    else:
        pass

# Validação Final
if imovel_atende_norma:
    print("Resultado: O imóvel rural atende a norma.")
else:
    print(f"Resultado: O imóvel rural não atende a norma porque houve {motivo_impedimento}")

**Exemplo com dados reais**

**Configuração e Exemplo Não atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/mini_ucs.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./mini_ucs.zip")
file_path = "./mini_ucs.zip/mini_ucs.shp"

mini_ucs = gpd.read_file(file_path)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_11.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_11.zip")
file_path = "./poligono_11.zip/poligono_11.shp"

poligono_11 = gpd.read_file(file_path)

In [ ]:
mini_ucs.tail()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_ucs.plot(ax=ax, color='#d62728', alpha=0.3)
mini_ucs.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_11.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_11.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Unidades de Conservação')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Unidades de Conservação')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
mini_ucs_plano = mini_ucs.to_crs(epsg=6933)
poligono_11_plano = poligono_11.to_crs(epsg=6933)

geom_empreendimento_plana = poligono_11_plano.geometry.iloc[0]

sobreposicoes = mini_ucs_plano.intersects(geom_empreendimento_plana)
areas = mini_ucs_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Unidade de Conservação {i}! Área da sobreposição: {areas[i]:.2f} m².")

**Configuração e Exemplo atende a norma**

In [ ]:
r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/poligono_12.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./poligono_12.zip")
file_path = "./poligono_12.zip/poligono_12.shp"

poligono_12 = gpd.read_file(file_path)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 8))

mini_ucs.plot(ax=ax, color='#d62728', alpha=0.3)
mini_ucs.boundary.plot(ax=ax, color='#d62728', linewidth=2)

poligono_12.plot(ax=ax, color='#1f77b4', alpha=0.3)
poligono_12.boundary.plot(ax=ax, color='#1f77b4', linewidth=2)

ax.set_aspect('equal')
ax.set_title('Visualização: Empreendimento vs. Unidades de Conservação')
ax.set_xlabel('Longitude / X')
ax.set_ylabel('Latitude / Y')
plt.grid(True, linestyle='--', alpha=0.6)

legenda_empreendimento = mpatches.Patch(color='#1f77b4', alpha=0.5, label='Empreendimento')
legenda_tis = mpatches.Patch(color='#d62728', alpha=0.5, label='Unidades de Conservação')
ax.legend(handles=[legenda_empreendimento, legenda_tis])
plt.show()

In [ ]:
mini_ucs_plano = mini_ucs.to_crs(epsg=6933)
poligono_12_plano = poligono_12.to_crs(epsg=6933)

geom_empreendimento_plana = poligono_12_plano.geometry.iloc[0]

sobreposicoes = mini_ucs_plano.intersects(geom_empreendimento_plana)
areas = mini_ucs_plano.intersection(geom_empreendimento_plana).area

area_minima = 1.0

for i, sobrepoe in sobreposicoes.items():
    if sobrepoe and areas[i] > area_minima:
        print(f"O empreendimento sobrepõe a área da Unidade de Conservação {i}! Área da sobreposição: {areas[i]:.2f} m².")